In [ ]:


import kagglehub
import os

path = kagglehub.dataset_download("shaunthesheep/microsoft-catsvsdogs-dataset")
print("Dataset path:", path)

dataset_path = os.path.join(path, "PetImages")
print(os.listdir(dataset_path))

Using Colab cache for faster access to the 'microsoft-catsvsdogs-dataset' dataset.
Dataset path: /kaggle/input/microsoft-catsvsdogs-dataset
['Dog', 'Cat']


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
import os
from PIL import Image

dataset_dir = os.path.join(path, 'PetImages')

bad_files = []

for class_name in os.listdir(dataset_dir):
    class_path = os.path.join(dataset_dir, class_name)

    if os.path.isdir(class_path):
        for file_name in os.listdir(class_path):
            file_path = os.path.join(class_path, file_name)
            try:
                with Image.open(file_path) as img:
                    img.verify()
            except Exception:
                bad_files.append(file_path)

print("Number of corrupted files:", len(bad_files))

for file_path in bad_files:
    try:
        os.remove(file_path)
    except:
        pass

print("Corrupted files removed.")

Number of corrupted files: 4
Corrupted files removed.


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(150,150),
    batch_size=32,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(150,150),
    batch_size=32,
    label_mode="binary"
)

train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
val_ds = val_ds.apply(tf.data.experimental.ignore_errors())

Found 25000 files belonging to 2 classes.
Using 20000 files for training.
Found 25000 files belonging to 2 classes.
Using 5000 files for validation.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# تجاهل الصور التالفة
train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
val_ds = val_ds.apply(tf.data.experimental.ignore_errors())

# بناء الموديل
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(150,150,3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 35s 53ms/step - accuracy: 0.5759 - loss: 0.7053 - val_accuracy: 0.7200 - val_loss: 0.5425
Epoch 2/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 31s 51ms/step - accuracy: 0.7149 - loss: 0.5592 - val_accuracy: 0.7624 - val_loss: 0.5057
Epoch 3/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.7753 - loss: 0.4718 - val_accuracy: 0.7942 - val_loss: 0.4663
Epoch 4/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.8262 - loss: 0.3898 - val_accuracy: 0.7950 - val_loss: 0.4567
Epoch 5/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 32s 51ms/step - accuracy: 0.8561 - loss: 0.3208 - val_accuracy: 0.7984 - val_loss: 0.4761
Epoch 6/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.8894 - loss: 0.2628 - val_accuracy: 0.8004 - val_loss: 0.5179
Epoch 7/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 52s 84ms/step - accuracy: 0.9073 - loss: 0.2175 - val_accuracy: 0.7903 - val_loss: 0.5791
Epoch 8/10
618/618 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.9252 - loss: 0.1809 - 

In [ ]:
loss, accuracy = model.evaluate(val_ds)

print("Accuracy:", accuracy)
print("Loss:", loss)

155/155 ━━━━━━━━━━━━━━━━━━━━ 42s 271ms/step - accuracy: 0.8007 - loss: 0.7070
Accuracy: 0.8012560606002808
Loss: 0.7155667543411255
